In [ ]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
!pip install transformers  accelerate
!pip install bitsandbytes

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM , BitsAndBytesConfig
import torch

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

In [ ]:
SYS_PROMPT = """You are an assistant for answering questions.
Only give the answer
Dont add any comments or extra explanations.
If you don't know the answer, just say "I do not know." Don't make up an answer.
Answer in Arabic if the question is in Arabic
"""

def format_prompt(question, context=""):
  PROMPT = f"My Question is : {question}\n"
  if context != "":
    PROMPT = PROMPT + f" You have to Answer in the following Context: {context}\n"
  return PROMPT

def get_answer(question, context=""):
  prompt = format_prompt(question, context)
  messages = [
      {"role": "system", "content": SYS_PROMPT},
      {"role": "user", "content": prompt}
  ]

  inputs = tokenizer.apply_chat_template(messages ,return_tensors='pt' , add_generation_prompt = True).to('cuda')
  pad_token_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else tokenizer.cos_token_id

  outputs = model.generate(
      inputs['input_ids'],
      max_new_tokens=50,
      pad_token_id=pad_token_id,
      return_dict_in_generate=True,
      do_sample=True,
      top_p=0.95,
      temperature=0.01,
      repetition_penalty=1.2
  )
  output = tokenizer.decode(outputs.sequences[0] , skip_special_tokens=True) # Corrected: Access sequences attribute
  return output

In [ ]:
def get_last_line(s):
  lines = s.splitlines()
  return lines[-1] if lines else ''

In [ ]:
question = "مامعنى اسم خالد?"
result = get_answer(question)
last_line = get_last_line(result)
print(last_line)

اسم خالد هو اسم عربي يعني "الخلود" أو "الموت".
